In [ ]:
import numpy as np
import pandas as pd
import csv
import json
import requests

# Importing publications

The publications from ESRF are imported.\
The publications were downloaded from the ESRF/ILL database:\

https://epn-library.esrf.fr/

## ESRF

There are four types of ESRF publications. We shall designate them as follows:
<br>
Type 1: Publications with ESRF authors and describing ESRF experiments
<br>
Type 2: Publications without ESRF authors and describing ESRF experiments
<br>
Type 3: Publications with ESRF authors and not describing ESRF experiments
<br>
Type 4: Articles citing the ESRF, no ESRF authors
<br>
This 'Type' information willl be included alongside the other metadata in the dataframes

In [ ]:
# Import all ESRF publication files
pub_esrf_type1=pd.read_csv('{insert pathname}/ESRF publications with ESRF authors and describing ESRF experiment (Oct 2024).csv',sep='	', encoding='utf-8',skiprows=1,header=0)    # Import file
pub_esrf_type2=pd.concat([pd.read_csv('{insert pathname}/ESRF Publications without ESRF authors and describing an ESRF experiment (2015 onwards).csv',sep='	', encoding='utf-8',skiprows=1,header=0),pd.read_csv('{insert pathname}/ESRF Publications without ESRF authors and describing an ESRF experiment (bef 2015).csv',sep='	', encoding='utf-8',skiprows=1,header=0)],ignore_index=True)    # Import file
pub_esrf_type3=pd.read_csv('{insert pathname}/ESRF Publications with ESRF authors and not describing an ESRF experiment.csv',sep='	', encoding='utf-8',skiprows=1,header=0)    # Import file
pub_esrf_type4=pd.read_csv('{insert pathname}/ESRF articles citing ESRF, no ESRF author.csv',sep='	', encoding='utf-8',skiprows=1,header=0)    # Import file

# Add 'Type' column
pub_esrf_type1['Type']=1
pub_esrf_type2['Type']=2
pub_esrf_type3['Type']=3
pub_esrf_type4['Type']=4

# Concatenate all four types
pub_esrf=pd.concat([pub_esrf_type1,pub_esrf_type2,pub_esrf_type3,pub_esrf_type4],ignore_index=True)

# Add 'Facility repository' column
pub_esrf['Facility repository']='ESRF'

# Uppercase all the proposal names
pub_esrf['Proposal number'] = pub_esrf['Proposal number'].str.upper()

In [ ]:
# Code to export the processed publications to a CSV file. Uncomment the following lines to run it.
# Checkpoint data

# pub_esrf.to_csv('{insert path here}/Publications_ESRF.csv',index=False)

In [ ]:
# Code to import the processed publications from a CSV file if needed. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/Publications_ESRF.csv'
# pub_esrf = pd.read_csv(filepath,encoding='utf-8')

# Get Proposals 
Now, that we have the publications, the next step is to get the proposals.\
For ESRF, this can be done through the API

## ESRF Proposals
The ESRF publications imported above are linked to their corresponding proposals through the proposal number.\
We want to get a complete list of ESRF proposals and map them to the corresponding papers.\
https://icatplus.esrf.fr/api/Documents provides a list of experimental sessions, which can then be mapped to the associated proposals and them to the assosicated publications.\
Note that multiple experimental sessions can be associated with a single proposal.

In [ ]:
# Import ESRF publications

filepath = '{insert pathname}/Publications_ESRF.csv'
pub_esrf=pd.read_csv(filepath)    # Import file

# Remove any publications without DOIs
pub_esrf=pub_esrf[pub_esrf['DOI'].notna()]

# Replace any NaN proposal names with empty string
pub_esrf['Proposal number']=pub_esrf['Proposal number'].fillna('')

# Convert all proposal names to uppercase
pub_esrf['Proposal number']=pub_esrf['Proposal number'].str.upper()

In [4]:
# Get list of experimental session documents
api_request="https://icatplus.esrf.fr/api/Documents/"
response=requests.get(api_request).json()
documents_esrf=response

In [5]:
# Extract DOI, Title, and Summary (abstract) information from documents_esrf into session_esrf
# session_esrf will be a list of dicts, with each index in the list representing a proposal; dict keys are: 'doi', 'title', 'summary'

session_esrf=[]        # Initialise empty session_esrf list

for document in documents_esrf:
    session_dict={}        # Initialise empty session_dict dictionary
    
    # Check for DOI and append those with one
    if document['doi']!=None:
        session_dict['doi']=document['doi']
        session_dict['title']=document['title']
        session_dict['summary']=document['summary']
        session_esrf.append(session_dict)
    else:
        pass

In [ ]:
# Function to fetch Proposal number information of a single session using API call

def fetch_data(session):
    api_request="https://icatplus.esrf.fr/doi/"
    response = requests.get(api_request+session['doi']+'/reports')

    # Check for valid API call status code
    if response.status_code==200:
        reports_esrf=response.json()
        proposals=[]
        for report in reports_esrf:
            proposals.append({report['proposal']:report['reports']})
        session['proposal_dict']=proposals
        return session     # prop with proposal number added 
    else:
        return 0        # return 0 if API call result is invalid

def fetch_subject(session):
    api_request="https://icatplus.esrf.fr/doi/"
    response2 = requests.get(api_request+session['doi']+'/json-datacite')

    # Check for valid API call status code
    if response2.status_code==200:
        reports2_esrf=response2.json()
        for subject in reports2_esrf['subjects']:
            if (subject['subjectScheme'] == 'Proposal Type Description') & ('subject' in subject):     # Check if the proposal type description exists, AKA the scientific disciplines such as 'Life Sciences' etc.
                session['subject']=subject['subject']
            elif (subject['subjectScheme'] == 'Instrument') & ('subject' in subject):     # Check if the instrument exists, AKA the beamline name
                session['instrument']=subject['subject']
            else:
                pass
        return session
    else:
        return 0

In [7]:
# Loop through each session in session_esrf and apply the fetch_data function. Only append session that has valid proposal number to session_esrf_valid

session_esrf_valid=[]
for session in session_esrf:
    fetched_data=fetch_data(session)
    if fetched_data!=0:
        session_esrf_valid.append(session)
    else:
        pass

In [24]:
# Loop through each session in session_esrf_valid and apply the fetch_subject function.

session_esrf_valid_with_subj=[]
for session in session_esrf_valid:
    fetched_data=fetch_subject(session)
    if fetched_data!=0:
        session_esrf_valid_with_subj.append(session)
    else:
        pass

In [ ]:
# Code to export the processed sessions with metadata to a JSON file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/session_esrf_valid_with_subj.json'
# with open(filepath, 'w') as f:
#     json.dump(session_esrf_valid_with_subj, f, indent=2)

In [ ]:
# Load JSON file containing processed sessions as a dictionary if needed
# Checkpoint data

# filepath = '{insert pathname}/session_esrf_valid_with_subj.json'    # <--- INSERT YOUR FILEPATH HERE
# with open(filepath, 'r') as f:
#         session_esrf_valid_with_subj = json.load(f)

In [141]:
# Check how many proposals numbers each session is linked to
sum_of_none = 0
sum_of_one = 0
sum_of_more = 0
for session in session_esrf_valid_with_subj:
    if len(session['proposal_dict']) == 1:
        sum_of_one += 1
    elif len(session['proposal_dict']) == 0:
        sum_of_none += 1
    else:
        sum_of_more += 1

print('Number of session with zero proposals',sum_of_none)
print('Number of session with exactly one proposal',sum_of_one)
print('Number of session with more than one proposals',sum_of_more)

Number of session with zero proposals 0
Number of session with exactly one proposal 8348
Number of session with more than one proposals 0


Seems like all sessions are linked to only one proposal. In that case, we can separate the proposal number from the pdf document name for each session.\
For example, 'proposal_dict': [{'MA-6393': ['109865_0001.pdf']}] can be split into 'proposal': 'MA-6393' and 'pdf document name': '109865_0001.pdf'


In [142]:
for session in session_esrf_valid_with_subj:
    session['proposal'] = list(session['proposal_dict'][0].keys())[0]
    
for session in session_esrf_valid_with_subj:
    session['pdf_document_name'] = list(session['proposal_dict'][0].values())[0]

In [143]:
# Convert session_esrf_valid to DataFrame, apply upper case to all proposal names for standardisation, replace all NaN values with blank strings in the 'subject' column.

df=pd.DataFrame(session_esrf_valid_with_subj)
df['proposal']=df['proposal'].str.upper()
df['subject']=df['subject'].fillna('')


# Group by proposal, and for each of the 'summary', 'title', 'subject' columns, concatenate unique strings. For 'doi' column, gather all the experiment session DOIs into a list for each proposal.
df = df.groupby('proposal').agg({
    'summary': lambda x: '. '.join(sorted(set(v for v in x if v))),
    'title': lambda x: '. '.join(set(x)),
    'subject': lambda x: ', '.join(sorted(set(v for v in x if v))),
    'instrument': lambda x: list(sorted(set(v for v in x if v))),
    'doi': lambda x: list(x.dropna()),
    'pdf_document_name': lambda x: list(set(i for sublist in x for i in sublist if i))
}).reset_index()

df.rename(columns={'doi':'experiment_session_doi'},inplace=True)

In [144]:
df

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name
0,A01-2-1247,The proposal falls within the general research...,Crystal structure of multiferroic KNi_1-xCo_xP...,,[BM01],[10.15151/ESRF-ES-670011307],[]
1,A01-2-1248,Having achieved successful results with metal-...,Metal-organic and covalent organic polyhedra f...,,[BM01],[10.15151/ESRF-ES-670011305],[]
2,A01-2-1249,The overall aim of the project is deciphering ...,The redox structure of haem- and flavoproteins...,,[BM01],[10.15151/ESRF-ES-670011413],[98064_A.pdf]
3,A01-2-1254,"In this study, we will investigate structural ...","Nickelates – phase transitions, distortions an...",,[BM01],[10.15151/ESRF-ES-748027553],[]
4,A01-2-1255,We developed a crystallization strategy that p...,Understanding the structure of two-dimensional...,,[BM01],[10.15151/ESRF-ES-670011338],[]
...,...,...,...,...,...,...,...
5379,XA-11,Steel slag is one of the most common wastes pr...,Time-resolved X-ray Tomography imaging and Ram...,,[BM05],[10.15151/ESRF-ES-1647778275],[]
5380,XA-5,The aim of this proposal is to elucidate the i...,ReMade Proposal\r\nImpact of metals blend and ...,,[ID22],[10.15151/ESRF-ES-1424924468],[]
5381,XA-6,Recycling spent Li-ion batteries has attracted...,ReMade Proposal\r\nOperando investigation stru...,,[ID31],[10.15151/ESRF-ES-1436201044],[]
5382,XA-7,This proposals combines our expertise in metal...,ReMade Proposal\r\nTuning the sorption propert...,,[ID31],[10.15151/ESRF-ES-1352264747],[]


In [ ]:
# Code to export the DataFrame with no references/publications metadata to a JSON file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/Proposals_ESRF_base.json'
# df.to_json(filepath, indent=2, orient='records')


### Proposals with PDF metadata only
Map PDF metadata obtained from GROBID to proposals. \
The PDF metadata we are interested in are the main body of text describing the proposal in greater detail, and the referenced works. \
Refer to Grobid.ipynb to see how PDF metadata was obtained.

In [ ]:
# Load json file containing proposals in base format
# Checkpoint data

filepath = '{insert pathname}/Proposals_ESRF_base.json'    # <--- INSERT YOUR FILEPATH HERE
df = pd.read_json(filepath)

In [4]:
# Initlialise empty lists for 'referenced_works DOI' and 'openalex_ids' columns
df['referenced_works_doi'] = np.empty((len(df), 0)).tolist()
df['openalex_ids'] = np.empty((len(df), 0)).tolist()

In [5]:
df

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name,referenced_works_doi,openalex_ids
0,A01-2-1247,The proposal falls within the general research...,Crystal structure of multiferroic KNi_1-xCo_xP...,,[BM01],[10.15151/ESRF-ES-670011307],[],[],[]
1,A01-2-1248,Having achieved successful results with metal-...,Metal-organic and covalent organic polyhedra f...,,[BM01],[10.15151/ESRF-ES-670011305],[],[],[]
2,A01-2-1249,The overall aim of the project is deciphering ...,The redox structure of haem- and flavoproteins...,,[BM01],[10.15151/ESRF-ES-670011413],[98064_A.pdf],[],[]
3,A01-2-1254,"In this study, we will investigate structural ...","Nickelates – phase transitions, distortions an...",,[BM01],[10.15151/ESRF-ES-748027553],[],[],[]
4,A01-2-1255,We developed a crystallization strategy that p...,Understanding the structure of two-dimensional...,,[BM01],[10.15151/ESRF-ES-670011338],[],[],[]
...,...,...,...,...,...,...,...,...,...
5379,XA-11,Steel slag is one of the most common wastes pr...,Time-resolved X-ray Tomography imaging and Ram...,,[BM05],[10.15151/ESRF-ES-1647778275],[],[],[]
5380,XA-5,The aim of this proposal is to elucidate the i...,ReMade Proposal\r\nImpact of metals blend and ...,,[ID22],[10.15151/ESRF-ES-1424924468],[],[],[]
5381,XA-6,Recycling spent Li-ion batteries has attracted...,ReMade Proposal\r\nOperando investigation stru...,,[ID31],[10.15151/ESRF-ES-1436201044],[],[],[]
5382,XA-7,This proposals combines our expertise in metal...,ReMade Proposal\r\nTuning the sorption propert...,,[ID31],[10.15151/ESRF-ES-1352264747],[],[],[]


In [ ]:
# Load json file containing PDF metadata as a dictionary; this file was created in the Grobid.ipynb
# Checkpoint data

filepath = '{insert pathname}/PDF_metadata.json'    # <--- INSERT YOUR FILEPATH HERE
with open(filepath, 'r') as f:
        pdf_dict_processed = json.load(f)

In [7]:
# Add OpenAlex URL to the IDs for correct formatting for input into model
for pdf in pdf_dict_processed:
    # text_list = pdf_dict_processed[pdf].get('text', [])
    # pdf_dict_processed[pdf]['text'] = ' '.join(filter(None, text_list))
    ls=[]
    for open_alex_id in pdf_dict_processed[pdf]['openalex_ids']:
        open_alex_id = 'https://openalex.org/' + open_alex_id
        ls.append(open_alex_id)
    pdf_dict_processed[pdf]['openalex_ids'] = ls

In [8]:
# Map each proposal to the corresponding PDF document names, concactenate all the referenced work DOIs, corresponsding OpenAlex IDs, and text from 
# the PDF metadata into the respective columns of the proposal DataFrame. Make sure the lists of DOIs and OpenAlex IDs do not have repeating elements.
prop_esrf_dict = df.to_dict(orient='records')
prop_esrf_pdf_only = []
for prop in prop_esrf_dict:
    for pdf in prop['pdf_document_name']:
        pdf = pdf.replace('.pdf', '')
        pdf_metadata = pdf_dict_processed.get(pdf)
        if pdf_metadata:
            prop['referenced_works_doi'] = list(set(prop['referenced_works_doi'] + pdf_metadata['referenced_works_doi']))
            prop['openalex_ids'] = list(set(prop['openalex_ids'] + pdf_metadata['openalex_ids']))
            # prop['referenced_works DOI'] = pdf_metadata['DOI']
            # prop['openalex_ids'] = pdf_metadata['OpenAlex']
            prop['summary'] = prop['summary'] + ' ' + pdf_metadata['text']
    prop_esrf_pdf_only.append(prop)

In [ ]:
# Code to export the processed proposals with PDF metadata to a JSON file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/Proposals_ESRF_PDF_metadata_only.json'
# with open(filepath, 'w') as f:
#     json.dump(prop_esrf_pdf_only, f, indent=2)

### Proposals with publications only
Map proposals to publications.

In [ ]:
# Load json file containing proposals in base format
# Checkpoint data

filepath = '{insert pathname}/Proposals_ESRF_base.json'    # <--- INSERT YOUR FILEPATH HERE
df = pd.read_json(filepath)

In [159]:
# Use Regex to map Proposal names in df to Proposal names in pub_esrf, and get a list of publication DOIs for each proposal
doi_list=[]
for proposal in df['proposal']:
    matching_dois=pub_esrf[pub_esrf['Proposal number'].str.contains(r'\b' + proposal + r'\b', regex=True,na=False)]['DOI'].dropna().to_list()
    doi_list.append(matching_dois)

# Append the list of publication DOIs to df under the new column 'publications DOI'
df['publications_doi']=doi_list

In [160]:
df

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name,publications_doi
0,A01-2-1247,The proposal falls within the general research...,Crystal structure of multiferroic KNi_1-xCo_xP...,,[BM01],[10.15151/ESRF-ES-670011307],[],[]
1,A01-2-1248,Having achieved successful results with metal-...,Metal-organic and covalent organic polyhedra f...,,[BM01],[10.15151/ESRF-ES-670011305],[],[]
2,A01-2-1249,The overall aim of the project is deciphering ...,The redox structure of haem- and flavoproteins...,,[BM01],[10.15151/ESRF-ES-670011413],[98064_A.pdf],[]
3,A01-2-1254,"In this study, we will investigate structural ...","Nickelates – phase transitions, distortions an...",,[BM01],[10.15151/ESRF-ES-748027553],[],[]
4,A01-2-1255,We developed a crystallization strategy that p...,Understanding the structure of two-dimensional...,,[BM01],[10.15151/ESRF-ES-670011338],[],[10.1038/s41563-023-01669-z]
...,...,...,...,...,...,...,...,...
5379,XA-11,Steel slag is one of the most common wastes pr...,Time-resolved X-ray Tomography imaging and Ram...,,[BM05],[10.15151/ESRF-ES-1647778275],[],[]
5380,XA-5,The aim of this proposal is to elucidate the i...,ReMade Proposal\r\nImpact of metals blend and ...,,[ID22],[10.15151/ESRF-ES-1424924468],[],[]
5381,XA-6,Recycling spent Li-ion batteries has attracted...,ReMade Proposal\r\nOperando investigation stru...,,[ID31],[10.15151/ESRF-ES-1436201044],[],[]
5382,XA-7,This proposals combines our expertise in metal...,ReMade Proposal\r\nTuning the sorption propert...,,[ID31],[10.15151/ESRF-ES-1352264747],[],[]


In [ ]:
# Code to export DataFrame to a JSON file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/Proposals_ESRF_publications_only_doi.json'    # <--- INSERT YOUR FILEPATH HERE
# df.to_json(filepath, orient='records', indent=2)

#### Get OpenAlex citation metadata for proposals
Now that we have the proposals with the DOIs of the publications linked to the proposals, we want to use the OpenAlex API call on the DOIs to get the corresponding OpenAlex IDs. This is because the topic classification model takes in OpenAlex IDs and not DOIs as input features.

In [ ]:
# Import ESRF proposals with publication DOIs only
# Checkpoint data

filepath = '{insert pathname}/Proposals_ESRF_publications_only_doi.json'   # <--- INSERT YOUR FILEPATH HERE
prop_esrf=pd.read_json(filepath)

In [163]:
prop_esrf

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name,publications_doi
0,A01-2-1247,The proposal falls within the general research...,Crystal structure of multiferroic KNi_1-xCo_xP...,,[BM01],[10.15151/ESRF-ES-670011307],[],[]
1,A01-2-1248,Having achieved successful results with metal-...,Metal-organic and covalent organic polyhedra f...,,[BM01],[10.15151/ESRF-ES-670011305],[],[]
2,A01-2-1249,The overall aim of the project is deciphering ...,The redox structure of haem- and flavoproteins...,,[BM01],[10.15151/ESRF-ES-670011413],[98064_A.pdf],[]
3,A01-2-1254,"In this study, we will investigate structural ...","Nickelates – phase transitions, distortions an...",,[BM01],[10.15151/ESRF-ES-748027553],[],[]
4,A01-2-1255,We developed a crystallization strategy that p...,Understanding the structure of two-dimensional...,,[BM01],[10.15151/ESRF-ES-670011338],[],[10.1038/s41563-023-01669-z]
...,...,...,...,...,...,...,...,...
5379,XA-11,Steel slag is one of the most common wastes pr...,Time-resolved X-ray Tomography imaging and Ram...,,[BM05],[10.15151/ESRF-ES-1647778275],[],[]
5380,XA-5,The aim of this proposal is to elucidate the i...,ReMade Proposal\r\nImpact of metals blend and ...,,[ID22],[10.15151/ESRF-ES-1424924468],[],[]
5381,XA-6,Recycling spent Li-ion batteries has attracted...,ReMade Proposal\r\nOperando investigation stru...,,[ID31],[10.15151/ESRF-ES-1436201044],[],[]
5382,XA-7,This proposals combines our expertise in metal...,ReMade Proposal\r\nTuning the sorption propert...,,[ID31],[10.15151/ESRF-ES-1352264747],[],[]


In [ ]:
from pyalex import Works, config
config.api_key = "<YOUR_API_KEY>"

In [165]:
# Function for applying the API call to each row of the proposal DataFrame prop_esrf

def get_openalex_id(row):
    # Get the 'publications DOI' column from the row
    publications_doi = row['publications_doi']
    openalex_id = []
    
    if len(publications_doi) != 0:
        for doi in publications_doi:
            result = Works().filter(doi=doi).select('id').get()
            # If the result is not empty, append the 'id', otherwise do nothing
            if result:
                openalex_id.append(result[0]['id'])
            else:
                pass
    else:
        pass
    
    row['openalex_ids']=openalex_id
   
    return row

In [166]:
# Apply the get_openalex_id to each row of prop_esrf
prop_esrf = prop_esrf.apply(get_openalex_id, axis=1)

In [167]:
# Sanity check
# Check if the number of publications DOI is equal to the number of OpenAlex IDs for each proposal
prop_esrf[prop_esrf['publications_doi'].apply(len)!=prop_esrf['openalex_ids'].apply(len)]

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name,publications_doi,openalex_ids
3512,LS-3076,Pathogenic mechanisms of absestos-related dise...,Release of metals and dissolution of mineral f...,Life Sciences,[ID21],[10.15151/ESRF-ES-744175308],[],[10.13133/2239-1002/18090],[]


There is a mismatch for one of the proposals. This particular publication DOI (10.13133/2239-1002/18090) does not exist in the OpenAlex data repository.

In [ ]:
# Export the proposals with publications only (now with OpenAlex IDs) to a JSON file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/Proposals_ESRF_publications_only.json'    # <--- INSERT YOUR FILEPATH HERE
# prop_esrf.to_json(filepath, orient='records', indent=2)

### Proposals with both PDF metadata and publications
Combine the two proposals

In [ ]:
# Import both files and concatenate them
# Checkpoint data

filepath1 = '{insert pathname}/Proposals_ESRF_PDF_metadata_only.json'   # <--- INSERT YOUR FILEPATH HERE
filepath2 = '{insert pathname}/Proposals_ESRF_publications_only.json'   # <--- INSERT YOUR FILEPATH HERE
df1=pd.read_json(filepath1)
df2=pd.read_json(filepath2)

In [11]:
df1

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name,referenced_works_doi,openalex_ids
0,A01-2-1247,The proposal falls within the general research...,Crystal structure of multiferroic KNi_1-xCo_xP...,,[BM01],[10.15151/ESRF-ES-670011307],[],[],[]
1,A01-2-1248,Having achieved successful results with metal-...,Metal-organic and covalent organic polyhedra f...,,[BM01],[10.15151/ESRF-ES-670011305],[],[],[]
2,A01-2-1249,The overall aim of the project is deciphering ...,The redox structure of haem- and flavoproteins...,,[BM01],[10.15151/ESRF-ES-670011413],[98064_A.pdf],[],[]
3,A01-2-1254,"In this study, we will investigate structural ...","Nickelates – phase transitions, distortions an...",,[BM01],[10.15151/ESRF-ES-748027553],[],[],[]
4,A01-2-1255,We developed a crystallization strategy that p...,Understanding the structure of two-dimensional...,,[BM01],[10.15151/ESRF-ES-670011338],[],[],[]
...,...,...,...,...,...,...,...,...,...
5379,XA-11,Steel slag is one of the most common wastes pr...,Time-resolved X-ray Tomography imaging and Ram...,,[BM05],[10.15151/ESRF-ES-1647778275],[],[],[]
5380,XA-5,The aim of this proposal is to elucidate the i...,ReMade Proposal\r\nImpact of metals blend and ...,,[ID22],[10.15151/ESRF-ES-1424924468],[],[],[]
5381,XA-6,Recycling spent Li-ion batteries has attracted...,ReMade Proposal\r\nOperando investigation stru...,,[ID31],[10.15151/ESRF-ES-1436201044],[],[],[]
5382,XA-7,This proposals combines our expertise in metal...,ReMade Proposal\r\nTuning the sorption propert...,,[ID31],[10.15151/ESRF-ES-1352264747],[],[],[]


In [12]:
df2

,proposal,summary,title,subject,instrument,experiment_session_doi,pdf_document_name,publications_doi,openalex_ids
0,A01-2-1247,The proposal falls within the general research...,Crystal structure of multiferroic KNi_1-xCo_xP...,,[BM01],[10.15151/ESRF-ES-670011307],[],[],[]
1,A01-2-1248,Having achieved successful results with metal-...,Metal-organic and covalent organic polyhedra f...,,[BM01],[10.15151/ESRF-ES-670011305],[],[],[]
2,A01-2-1249,The overall aim of the project is deciphering ...,The redox structure of haem- and flavoproteins...,,[BM01],[10.15151/ESRF-ES-670011413],[98064_A.pdf],[],[]
3,A01-2-1254,"In this study, we will investigate structural ...","Nickelates – phase transitions, distortions an...",,[BM01],[10.15151/ESRF-ES-748027553],[],[],[]
4,A01-2-1255,We developed a crystallization strategy that p...,Understanding the structure of two-dimensional...,,[BM01],[10.15151/ESRF-ES-670011338],[],[10.1038/s41563-023-01669-z],[https://openalex.org/W4386923995]
...,...,...,...,...,...,...,...,...,...
5379,XA-11,Steel slag is one of the most common wastes pr...,Time-resolved X-ray Tomography imaging and Ram...,,[BM05],[10.15151/ESRF-ES-1647778275],[],[],[]
5380,XA-5,The aim of this proposal is to elucidate the i...,ReMade Proposal\r\nImpact of metals blend and ...,,[ID22],[10.15151/ESRF-ES-1424924468],[],[],[]
5381,XA-6,Recycling spent Li-ion batteries has attracted...,ReMade Proposal\r\nOperando investigation stru...,,[ID31],[10.15151/ESRF-ES-1436201044],[],[],[]
5382,XA-7,This proposals combines our expertise in metal...,ReMade Proposal\r\nTuning the sorption propert...,,[ID31],[10.15151/ESRF-ES-1352264747],[],[],[]


In [13]:
# Rename the 'openalex_ids' columns in both DataFrames to avoid confusion, and then concatenate them
df1.rename(columns={'openalex_ids':'referenced_works_openalex_ids'},inplace=True)
df2.rename(columns={'openalex_ids':'publications_openalex_ids'},inplace=True)

df_combined = df1.merge(df2[['proposal','publications_doi','publications_openalex_ids']], on='proposal', how='inner')

In [14]:
df_combined['combined_openalex_ids'] = df_combined.apply(lambda row: set(row['referenced_works_openalex_ids'] + row['publications_openalex_ids']), axis=1)

In [ ]:
# Export the proposals with combined publications and referenced work Openalex IDs to a JSON file. Uncomment the following lines to run it.
# Checkpoint data

# filepath = '{insert pathname}/Proposals_ESRF_combined.json'    # <--- INSERT YOUR FILEPATH HERE
# df_combined.to_json(filepath, orient='records', indent=2)